In [1]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API key: ")

if "JINA_API_KEY" not in os.environ:
    os.environ["JINA_API_KEY"] = getpass("Enter your JINA API key: ")

In [2]:
from typing import TypedDict, List, Tuple

import requests
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# Switch to "openai/gpt-oss-120b" if you want the newer heavyweight Groq model.
LLM_MODEL = "llama-3.3-70b-versatile"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
JINA_RERANKER_MODEL = "jina-reranker-v2-base-multilingual"
TOP_K = 8         # how many chunks the retriever pulls
TOP_N = 4         # how many chunks finally go to the LLM

llm = ChatGroq(model=LLM_MODEL, temperature=0)

/var/folders/nd/pvtypqp503l5pczkpc8ww5wh0000gp/T/ipykernel_18719/2682331933.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
DOCS = [
    # --- Decoys: short, keyword-saturated, no mechanistic answer ---
    "The attention mechanism is the core innovation in transformer architectures.",
    "Attention helps transformer models handle long-range dependencies in text sequences.",
    "Self-attention is a key building block of modern language models like BERT and GPT.",
    "Multi-head attention runs several attention computations in parallel and concatenates the outputs.",
    # --- Stronger decoys: mirror the query's vocabulary closely ---
    "The attention mechanism computes relationships between every pair of tokens to capture the context of the input.",
    "Token relationships are learned through attention layers during model training.",

    # --- THE ACTUAL ANSWER (long, mechanistic, dense) ---
    (
        "Self-attention works by projecting each input embedding into three learned vectors: "
        "a Query (Q), a Key (K) and a Value (V). For every pair of positions in the sequence, "
        "the model takes the dot product of one position's Query with another position's Key, "
        "divides by the square root of the embedding dimension to stabilize gradients, and "
        "applies softmax across positions to obtain attention weights. The output for each "
        "position is then a weighted sum of all V vectors, where the weights are the softmaxed "
        "dot products. This is why the operation is also called scaled dot-product attention."
    ),

    # --- Off-topic noise ---
    "Positional encodings are added so that transformers know the order of tokens in the input.",
    "Layer normalization is applied to stabilize training dynamics in deep transformer networks.",
    "Tokenization breaks input text into subword units before any embedding lookup happens.",
]

ANSWER_DOC_IDX = 6  # index of the truly relevant chunk — used later for tracking its rank

QUERY = "How does the attention mechanism actually compute the relationship between tokens?"

print(f"Total chunks in corpus : {len(DOCS)}")
print(f"Truly relevant chunk   : index {ANSWER_DOC_IDX}")

Total chunks in corpus : 10
Truly relevant chunk   : index 6


In [4]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

vectorstore = Chroma.from_texts(
    texts=DOCS,
    embedding=embeddings,
    metadatas=[{"source_idx": i} for i in range(len(DOCS))],
    collection_name="rerank_demo",
)
print("Vector store built.")

/Users/ankit.mishra/ankit/ML_prep/vecless/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1126 11:31:59.028000 18719 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8573.39it/s]


Vector store built.


In [5]:
class BasicRAGState(TypedDict):
    query: str
    retrieved: List[Tuple[Document, float]]   # (doc, similarity score)
    answer: str

RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a precise technical assistant. Answer the question using ONLY the context below. "
    "If the context does not contain the answer, say so.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def retrieve_node(state: BasicRAGState) -> dict:
    docs_and_scores = vectorstore.similarity_search_with_score(state["query"], k=TOP_K)
    return {"retrieved": docs_and_scores}

def generate_node(state: BasicRAGState) -> dict:
    top_docs = state["retrieved"][:TOP_N]
    context = "\n\n---\n\n".join(d.page_content for d, _ in top_docs)
    chain = RAG_PROMPT | llm
    response = chain.invoke({"context": context, "question": state["query"]})
    return {"answer": response.content}

builder = StateGraph(BasicRAGState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
basic_rag = builder.compile()

print("Basic RAG graph compiled.")

Basic RAG graph compiled.


In [8]:
result_basic = basic_rag.invoke({"query": QUERY, "retrieved": [], "answer": ""})

print("=" * 80)
print(f"QUERY: {QUERY}")
print("=" * 80)
print("\nTop-K retrieved by the bi-encoder (lower distance = more similar):\n")

for rank, (doc, score) in enumerate(result_basic["retrieved"], 1):
    idx = doc.metadata.get("source_idx")
    is_answer = " ⭐ (truly relevant)" if idx == ANSWER_DOC_IDX else ""
    preview = doc.page_content[:90].replace("\n", " ") + "..."
    print(f"  Rank {rank} | dist={score:.4f} | idx={idx}{is_answer}")
    print(f"           {preview}")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
JINA_RERANK_URL = "https://api.jina.ai/v1/rerank"

def jina_rerank(query: str, documents: List[str], top_n: int = TOP_K, model: str = JINA_RERANKER_MODEL):
    """Call Jina Reranker API and return list of {index, relevance_score, text}."""
    response = requests.post(
        JINA_RERANK_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ['JINA_API_KEY']}",
        },
        json={
            "model": model,
            "query": query,
            "top_n": top_n,
            "documents": documents,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["results"]

# Pull the docs out of the bi-encoder's top-K result
candidate_docs = [doc.page_content for doc, _ in result_basic["retrieved"]]
candidate_indices = [doc.metadata["source_idx"] for doc, _ in result_basic["retrieved"]]

reranked = jina_rerank(QUERY, candidate_docs, top_n=TOP_K)

print("=" * 80)
print("After cross-encoder reranking (higher score = more relevant):")
print("=" * 80)
for rank, r in enumerate(reranked, 1):
    # r["index"] is the position in the input documents list, NOT the original corpus idx
    original_idx = candidate_indices[r["index"]]
    is_answer = " ⭐ (truly relevant)" if original_idx == ANSWER_DOC_IDX else ""
    preview = candidate_docs[r["index"]][:90].replace("\n", " ") + "..."
    print(f"  Rank {rank} | score={r['relevance_score']:.4f} | idx={original_idx}{is_answer}")
    print(f"           {preview}")

# ReRanker implemented inside 

Now we wire reranking into the graph itself: `retrieve → rerank → generate`. The state grows by one field, the graph by one node.

In [ ]:
class RerankRAGState(TypedDict):
    query: str
    retrieved: List[Document]
    reranked: List[Document]
    answer: str

def retrieve_node_v2(state: RerankRAGState) -> dict:
    docs = vectorstore.similarity_search(state["query"], k=TOP_K)
    return {"retrieved": docs}

def rerank_node(state: RerankRAGState) -> dict:
    docs = state["retrieved"]
    texts = [d.page_content for d in docs]
    results = jina_rerank(state["query"], texts, top_n=TOP_N)

    reranked_docs = []
    for r in results:
        original_doc = docs[r["index"]]
        reranked_docs.append(
            Document(
                page_content=original_doc.page_content,
                metadata={
                    **original_doc.metadata,
                    "rerank_score": r["relevance_score"],
                },
            )
        )
    return {"reranked": reranked_docs}

def generate_node_v2(state: RerankRAGState) -> dict:
    context = "\n\n---\n\n".join(d.page_content for d in state["reranked"])
    chain = RAG_PROMPT | llm
    response = chain.invoke({"context": context, "question": state["query"]})
    return {"answer": response.content}

builder2 = StateGraph(RerankRAGState)
builder2.add_node("retrieve", retrieve_node_v2)
builder2.add_node("rerank", rerank_node)
builder2.add_node("generate", generate_node_v2)
builder2.set_entry_point("retrieve")
builder2.add_edge("retrieve", "rerank")
builder2.add_edge("rerank", "generate")
builder2.add_edge("generate", END)
rerank_rag = builder2.compile()

print("Reranker-augmented RAG graph compiled.")
